# Ylivertainen v2 — Clinical Association Pipeline

End-to-end workflow for finding statistically valid associations between **target outcomes**
and **predictor variables** in a clinical dataset.

This notebook drives six universal modules:

| Module                       | Purpose                                                    |
|------------------------------|------------------------------------------------------------|
| `schema_infer.py`            | Auto-classify each column (continuous, ordinal, …)         |
| `cleaning.py`                | Apply the schema, audit duplicates, derive new columns     |
| `dda.py`                     | Per-column descriptive stats + SVG plots                   |
| `missingness_resolution.py`  | Missing pattern analysis, flags, MICE multiple imputation  |
| `eda.py`                     | Univariate target × predictor screening (FDR-corrected)    |
| `inferential.py`             | Multivariable logistic regression with Rubin pooling       |

**Pipeline order**

```
load → infer schema → clean → DDA → missingness → derive new cols → DDA again
   → EDA screen → MICE impute → multivariable logistic (Rubin pool) → outputs
```

All outputs land under `output/<stage>/{figures,tables}/` as SVG and CSV.


## 0. Setup

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)

import numpy as np
import shutil
from pathlib import Path

from schema_infer import (infer_schema, print_schema_template, print_column_uniques, schema_summary,
                          export_schema_summary, ColSpec)
from cleaning import (apply_schema, audit_duplicates, export_cleaning_artifacts,
                      write_cleaned_csv, bin_numeric, bin_datetime, make_missing_flag,
                      combine_categories)
from dda import run_dda, plot_distribution_by_year
from missingness_resolution import (analyze_missingness, add_missing_flags,
                                    mark_structural_missing, drop_rows,
                                    mice_impute, simple_impute, imputation_audit)
from eda import screen_associations
from inferential import run_inferential, summarize_multivariable_cases

OUTPUT_ROOT = Path("output")
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

from config import load

#🟧🟧🟧 None = all years; e.g. [2025] for one cohort year

ANALYSIS_YEARS: list[int] | None = None

# Run top to bottom

Each section: **edit → run → next**. No jumping back to a config block at the top.


## 01. Load data


In [ ]:
DATA_PATH = "Meningiomas PSKUS grants - Visi pacienti.csv"   # or "yourdata.csv"

_c01 = load("01_cohort")
df_raw = _c01.load_raw(DATA_PATH)
df_raw.head(0)


## 02. Column rename

1. Run **see raw columns** → copy the printed skeleton  
2. Paste into `COLUMN_RENAME_MAP` below and fill in snake_case names  
3. Run **apply rename**


In [ ]:
#🟧🟧🟧 Step 1 — see raw columns (run once per new dataset)

load("02_column_rename_map").list_cols(df_raw)


In [ ]:
#🟧🟧🟧 Step 2 — paste skeleton here and fill in the right-hand names

COLUMN_RENAME_MAP = {
    "Nr.": "id",
    "Personas kods": "patient_code",
    "Unnamed: 2": "entry_year",
    "Vecums. gadi": "age",
    "Dzimums. 0 - vīrietis\n1 - sieviete\"": "sex",
    "Histoloģija. 0 - nav\n1 - ir": "histology_available",
    "WHO pakāpe (2021). 1 / 2 / 3": "who_grade",
    "Progesterons. 0 - negatīvs\n1 - pozitīvs": "progesterone_pos",
    "Ki-67 (%). skaitlis. %": "ki67_pct",
    "Smadzeņu parenhīmas invāzija. 0 - nav\n1 - ir": "brain_invasion",

    "Nekroze histoloģiski. 0 - nav\n1 - ir": "hist_necrosis",
    "MRI izmeklējuma datums": "mri_date",
    "Puse. 1 - labā\n2 - kreisā\n3 - viduslīnija": "side",
    "Lokalizācija: skull base / non–skull base. 0 - non-skull base\n1 - skull base": "tumor_location",
    "Cik meningiomas?": "meningioma_count",

    "Max diametrs. skaitlis.cm": "max_diameter_cm",
    "Tilpums": "tumor_volume",
    "Pamatmodalitāte analīzei. 0 - MRI\n1 - CT\n3 - MRI+CT": "base_modality",
    "K/v i/v. 0 - nav\n1 - ir": "iv_contrast",
    "0 - primārs\n1 - recidīvs": "tumor_episode",
    "Audzēja robeža. 1 = gluda. \n2 = neregulāra": "tumor_margin",
    "Dural tail sign. 0 - nav\n1 - ir": "dural_tail",
    "Gredzenveida kontrastēšanās (tumor capsular enhancement). 0 - nav\n1 - ir": "capsular_enhancement",
    "Kontrastēšanās veids. 0 - homogēna\n1 - heterogēna": "heterogeneous_enhancement",
    "Perifokāla tūska. 0 - nav\n1 - ir": "perifocal_edema",

    "Perifokālas tūskas tilpums. cm3": "edema_volume_cm3",
    "Masas efekts. 0 - nav\n1 - ir": "mass_effect",
    "Audzēja kalcifikācija. 0 - nav\n1 - ir": "calcification",
    "Cistiskas komponentes. 0 - nav\n1 - ir": "cystic_component",
    "Audzēja nekroze. 0 - nav\n1 - ir": "necrosis",
    "Hemorāģiskas sastāvdaļas. 0 - nav\n1 - ir\n2 - nav skaidri izvērtējams ": "hemorrhage",
    "Kaule hiperostoze. 0 - nav\n1 - ir": "hyperostosis",
    "Kaula invāzija (cortical destruction). 0 - nav\n1 - ir": "cortical_destruction",
    "Tumor Hyperintensity on DWI. 0 - nav\n1 - ir": "dwi_hyperintensity",
    "Tumor Hyperintensity on T2. 0 - nav\n1 - ir": "t2_hyperintensity",
    "Tumor Hypointensity on T1. 0 - nav\n1 - ir": "t1_hypointensity",

    "Sīnuss. 0 - neieaug\n1 - ieaug\n2 - ieaug un cauraug": "sinus_invasion",
    "Cauraug falx cerebri 0 - nav. 1 - ir": "transfalcine_extension",
    "ADC map value": "adc_value",
}

In [ ]:
#🟧🟧🟧 Step 3 — apply rename

df_raw = load("02_column_rename_map").apply_rename(df_raw, COLUMN_RENAME_MAP)
df = df_raw
df.head(0)

## 02b. Key columns & cohort filter

Use **renamed** names from §02. Edit, then run.


In [ ]:
YEAR_COLUMN = "entry_year"
ID_COLS = ["id", "patient_code", "entry_year"]

ANALYSIS_YEARS: list[int] | None = None   # e.g. [2025]; None = all years


In [ ]:
df_raw = _c01.filter_cohort(df_raw, YEAR_COLUMN, ANALYSIS_YEARS)
df = df_raw

## 03. Schema

1. Run **infer**  
2. Run **print template**  
3. Run **column uniques** (nulls / replace hints)  
4. Edit **schema_overrides**  
5. Run **apply overrides**


In [ ]:
schema = infer_schema(df_raw)
schema_summary(schema)


In [ ]:
print_schema_template(schema)


In [ ]:
#🟧🟧🟧 inspect raw values — use for nulls=() and replace={} below
print_column_uniques(df_raw, schema)


In [ ]:
#🟧🟧🟧 Edit overrides, then run

schema_overrides = {
    'id': ColSpec(name='id', kind='id'),
    'patient_code': ColSpec(name='patient_code', kind="id", keep=False),
    'entry_year': ColSpec(name='entry_year', kind='datetime', keep=False, datetime_bin='year'),
    'age': ColSpec(name='age', kind='continuous'),
    'sex': ColSpec(name='sex', kind='nominal', replace={0:"male", 1:"female",}),
    'histology_available': ColSpec(name='histology_available', kind='binary', nulls=(2,)),
    'who_grade': ColSpec(name='who_grade', kind='ordinal', ordered_levels=["1","2","3"]),
    'progesterone_pos': ColSpec(name='progesterone_pos', kind='binary', nulls=(2,)),
    'ki67_pct': ColSpec(name='ki67_pct', kind='text'),
    'brain_invasion': ColSpec(name='brain_invasion', kind='binary'),
    'hist_necrosis': ColSpec(name='hist_necrosis', kind='binary'),
    'mri_date': ColSpec(name='mri_date', kind='datetime', keep=False, datetime_bin='full'),
    'side': ColSpec(name='side', kind='nominal', replace={1: "right", 2: "left", 3: "midline"}),
    'tumor_location': ColSpec(name='tumor_location', kind='nominal', replace={0: "non_skull_base", 1: "skull_base"}, nulls=(2,)),
    'meningioma_count': ColSpec(name='meningioma_count', kind='ordinal', ordered_levels=[1,2,3,4,5]),
    'max_diameter_cm': ColSpec(name='max_diameter_cm', kind='continuous'),
    'tumor_volume': ColSpec(name='tumor_volume', kind='continuous'),
    'base_modality': ColSpec(name='base_modality', kind='nominal', replace={0: "mri", 1: "ct", 3: "mri_ct"}),
    'iv_contrast': ColSpec(name='iv_contrast', kind='binary'),
    'tumor_episode': ColSpec(name='tumor_episode', kind='ordinal', replace={'0': "primary", '1': "recurrent"}, ordered_levels=["primary", "recurrent"], nulls=("multiplas",)),
    'tumor_margin': ColSpec(name='tumor_margin', kind='nominal', replace={1: "regular", 2: "irregular"}, nulls=(0,)),
    'dural_tail': ColSpec(name='dural_tail', kind='binary'),
    'capsular_enhancement': ColSpec(name='capsular_enhancement', kind='binary'),
    'heterogeneous_enhancement': ColSpec(name='heterogeneous_enhancement', kind='binary'),
    'perifocal_edema': ColSpec(name='perifocal_edema', kind='binary'),
    'edema_volume_cm3': ColSpec(name='edema_volume_cm3', kind='continuous'),
    'mass_effect': ColSpec(name='mass_effect', kind='binary'),
    'calcification': ColSpec(name='calcification', kind='binary'),
    'cystic_component': ColSpec(name='cystic_component', kind='binary'),
    'necrosis': ColSpec(name='necrosis', kind='binary'),
    'hemorrhage': ColSpec(name='hemorrhage', kind='binary', nulls=(2.0,)),
    'hyperostosis': ColSpec(name='hyperostosis', kind='binary'),
    'cortical_destruction': ColSpec(name='cortical_destruction', kind='binary'),
    'dwi_hyperintensity': ColSpec(name='dwi_hyperintensity', kind='binary', nulls=('-',)),
    't2_hyperintensity': ColSpec(name='t2_hyperintensity', kind='binary', nulls=('-',)),
    't1_hypointensity': ColSpec(name='t1_hypointensity', kind='binary', nulls=('-',)),
    'sinus_invasion': ColSpec(name='sinus_invasion', kind='ordinal', replace={0: "no_invasion", 1: "sinus_invasion", 2: "transsinus_extension"}, ordered_levels=["no_invasion", "sinus_invasion", "transsinus_extension"]),
    'transfalcine_extension': ColSpec(name='transfalcine_extension', kind='binary'),
    'adc_value': ColSpec(name='adc_value', kind='continuous'),
    }


In [ ]:
load("03_schema_overrides").apply_schema_overrides(schema, schema_overrides, OUTPUT_ROOT)

## 04. Apply schema


In [ ]:
schema_log = []
df = apply_schema(df_raw, schema, log=schema_log)
n_rows_after_schema = len(df)
df.head()

## 05. Duplicate audit


In [ ]:
dupes, df = audit_duplicates(df, id_cols=ID_COLS, include_first=True, drop=False)
dupes.head() if len(dupes) else print('No duplicate groups found.')

## 06. Row filters

Edit `row_filters` (toggle `active=True/False`), run the cell, then run finalize below.


In [ ]:
_c04 = load("04_row_filters")

row_filters = [
    _c04.RowFilter(
        name="who_grade not NaN",
        keep=lambda d: d["who_grade"].notna(),
        note="TARGET column",
        active=True,
    ),
    # _c04.RowFilter(
    #     name="sex known",
    #     keep=lambda d: d["sex"] != "unknown",
    #     note="Keep rows where sex is known (not 'unknown')",
    #     active=False,
    # ),
    # _c04.RowFilter(
    #     name="adult patients only",
    #     keep=lambda d: d["age"] >= 18,
    #     note="Keep rows where age is 18 or older",
    #     active=False,
    # ),
    # _c04.RowFilter(
    #     name="exclude WHO grade 2 or 3",
    #     keep=lambda d: ~d["who_grade"].isin(["2", "3"]),
    #     note="Keep rows where WHO grade is 1 (exclude grades 2 and 3)",
    #     active=False,
    # ),
]

df, row_filter_log = _c04.apply_row_filters(df, row_filters)
row_filter_log


In [ ]:
df = _c04.finalize_row_drops(
    df, row_filter_log,
    output_root=OUTPUT_ROOT,
    df_raw=df_raw,
    n_rows_after_schema=n_rows_after_schema,
    schema=schema,
    dupes=dupes,
    schema_log=schema_log,
)


## 07. DDA — first pass


In [ ]:
from cleaning import format_table_for_display
from IPython.display import display

dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)

for name in ("overall", "continuous", "categorical", "binary", "datetime", "id_text"):
    print(f"\n--- {name} ---")
    tbl = dda_tables.get(name)
    if tbl is None or tbl.empty:
        print("(none)")
    else:
        display(format_table_for_display(tbl))


## 08. Missingness analysis


In [ ]:
missing_summary = analyze_missingness(df, output_root=OUTPUT_ROOT)
missing_summary.head(10)

### 08a. Missingness policy

Declare structural and MNAR decisions in lists below (same pattern as row filters).
Run the next cell to apply them once and get an audit log.

- **Structural** — NaN means the slot does not exist (do not impute; derive count/max instead).
- **MNAR** — missingness itself may be informative (adds `<col>_missing` flag).


In [ ]:
_c05 = load("05_missingness")

STRUCTURAL_GROUPS = [
    # Example only. Keep empty if we do not currently have slot-style columns.
    # _c05.StructuralGroup(
    #     name="lesion_mri_pirads",
    #     cols=["lesion_1_MRI_PIRADS", "lesion_2_MRI_PIRADS", "lesion_3_MRI_PIRADS"],
    #     derive_count_col="n_mri_pirads_lesions",
    #     derive_max_col="max_mri_pirads",
    #     skip_raw=True,
    #     reason="Blank lesion slots mean lesion does not exist, not unknown.",
    # ),
]

MNAR_COLUMNS = [
    # Add only when missingness itself may be informative.
    # _c05.MnarColumn(
    #     col="ki67_pct",
    #     flag_col="ki67_pct_missing",
    #     reason="Ki-67 may be absent because it was not measured/reported in selected cases.",
    # ),
    # _c05.MnarColumn(
    #     col="adc_value",
    #     flag_col="adc_value_missing",
    #     reason="ADC may be absent when DWI/ADC was unavailable or non-diagnostic.",
    # ),
]


In [ ]:
df, schema, missingness_log = _c05.apply_missingness_policy(
    df=df,
    schema=schema,
    structural_groups=STRUCTURAL_GROUPS,
    mnar_columns=MNAR_COLUMNS,
)

missingness_log

## 09. Derivations

Declare derived columns in a list below (same pattern as row filters / missingness).
All study-specific logic lives in the notebook; `06_derivations.py` is just the engine.

**Building blocks:**

- **`BinNumeric`** — cut a numeric column into ordered bins (`age` → `age_bins`).
  Bins = edge values; labels = one per gap. `len(bins) - 1 == len(labels)`.
  Default `right=False`: left-closed intervals (`[50, 60)` → `"50-59"`).
- **`IsIn`** — boolean flag when source value is in a list (`who_grade` in `["2","3"]` → `high_grade`).
- **`Apply`** — custom logic via helper + `fn=lambda s: ...` (Ki-67 midpoint, grouped labels, etc.).

Each entry supports `active=False` (skip) and `overwrite=True` (replace existing column).
Append to `DERIVATIONS` to add columns — no `.py` edits needed.


In [ ]:
_c06 = load("06_derivations")


def _ki67_midpoint(x):
    if pd.isna(x):
        return pd.NA
    parts = str(x).replace(",", ".").split("-")
    nums = [float(p) for p in parts]
    return sum(nums) / len(nums)
def _ki67_group(x):
    if pd.isna(x):
        return pd.NA
    if x <= 4:
        return "low_le_4"
    if x < 10:
        return "intermediate_5_9"
    return "high_ge_10"


DERIVATIONS = [
    _c06.BinNumeric(
        name="age_bins",
        source="age",
        bins=[-np.inf, 50, 60, 70, 80, np.inf],
        labels=["<50", "50-59", "60-69", "70-79", "80+"],
        kind="ordinal",
        active=True,
        overwrite=False,
        reason="Age groups for descriptive tables.",
    ),
    _c06.IsIn(
        name="high_grade",
        source="who_grade",
        values=["2", "3"],
        kind="binary",
        active=True,
        overwrite=False,
        reason="WHO grade 2/3 = high-grade meningioma.",
    ),
    _c06.Apply(
        name="ki67_mid",
        source="ki67_pct",
        fn=lambda s: s.map(_ki67_midpoint).astype("Float64"),
        kind="continuous",
        active=True,
        overwrite=False,
        reason="Midpoint of Ki-67 range strings.",
    ),
    _c06.Apply(
        name="ki67_group",
        source="ki67_mid",
        fn=lambda s: s.map(_ki67_group),
        kind="ordinal",
        ordered_levels=["low_le_4", "intermediate_5_9", "high_ge_10"],
        active=True,
        overwrite=False,
        reason="Ki-67 clinical groups: ≤4 / 5-9 / ≥10.",
    ),
    _c06.Compute(
        name="edema_volume_cm3",
        sources=["perifocal_edema", "edema_volume_cm3"],
        fn=lambda d: d["edema_volume_cm3"].mask(
            d["perifocal_edema"].fillna(1).astype(float) == 0, 0
        ),
        kind="continuous",
        active=True,
        overwrite=True,
        reason="No perifocal edema => edema volume is structurally 0, not missing.",
    ),
]


In [ ]:
df, schema, derivation_log = _c06.apply_derivations(
    df=df,
    schema=schema,
    derivations=DERIVATIONS,
    output_root=OUTPUT_ROOT,
    write_csv=True,
)

derivation_log


## 10. DDA — second pass


In [ ]:
from cleaning import format_table_for_display
from IPython.display import display

dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)

for name in ("overall", "continuous", "categorical", "binary", "datetime", "id_text"):
    print(f"\n--- {name} ---")
    tbl = dda_tables.get(name)
    if tbl is None or tbl.empty:
        print("(none)")
    else:
        display(format_table_for_display(tbl))


## 11. Analysis targets & predictors

Edit lists, then run.


In [ ]:
EDA_TARGETS = ['high_grade']
EDA_PREDICTORS = ['age', 'sex', 'histology_available', 'progesterone_pos', 'brain_invasion',
       'hist_necrosis', 'side', 'tumor_location',
       'meningioma_count', 'max_diameter_cm', 'tumor_volume', 'base_modality',
       'iv_contrast', 'tumor_episode', 'tumor_margin', 'dural_tail',
       'capsular_enhancement', 'heterogeneous_enhancement', 'perifocal_edema',
       'edema_volume_cm3', 'mass_effect', 'calcification', 'cystic_component',
       'necrosis', 'hemorrhage', 'hyperostosis', 'cortical_destruction',
       'dwi_hyperintensity', 't2_hyperintensity', 't1_hypointensity',
       'sinus_invasion', 'transfalcine_extension', 'adc_value', 'ki67_mid',
       'ki67_group', 'age_bins']

INFERENTIAL_TARGETS = ['high_grade']
INFERENTIAL_PREDICTORS = [
    #'sex',
    #'histology_available',
    #'progesterone_pos',
    #'brain_invasion',
    #'hist_necrosis',
    #'side',
    'tumor_location',
    #'meningioma_count',
    #'max_diameter_cm',
    'tumor_volume',
    #'base_modality',
    #'iv_contrast',
    #'tumor_episode',
    #'tumor_margin',
    #'dural_tail',
    #'capsular_enhancement',
    #'heterogeneous_enhancement',
    'perifocal_edema',
    #'edema_volume_cm3',
    #'mass_effect',
    #'cystic_component',
    #'necrosis',
    #'hemorrhage',
    'hyperostosis',
    #'cortical_destruction',
    #'dwi_hyperintensity',
    #'t2_hyperintensity',
    #'t1_hypointensity',
    #'sinus_invasion',
    #'transfalcine_extension',
    #'adc_value',
    #'ki67_pct',  #not gonna be analysed anyway
    #'ki67_mid',
    #'ki67_group',
    #'age_bins',
    #'age',
    ]


In [ ]:
_c07 = load("07_analysis")
(
    EDA_TARGETS,
    EDA_PREDICTORS,
    INFERENTIAL_TARGETS,
    INFERENTIAL_PREDICTORS,
    EDA_POSITIVE_CLASS,
    INFERENTIAL_POSITIVE_CLASS,
) = _c07.resolve_analysis(
    df,
    EDA_TARGETS,
    EDA_PREDICTORS,
    INFERENTIAL_TARGETS,
    INFERENTIAL_PREDICTORS,
)

df.tumor_episode.value_counts()

## 10. EDA — univariate screening

Targets can be **binary**, **continuous**, **ordinal**, or **nominal** (from schema). The test depends on both outcome and predictor types — e.g. ordinal outcome × nominal predictor → χ²; continuous outcome × nominal predictor → Kruskal–Wallis.

| target kind   | continuous / count predictor | ordinal predictor | nominal / binary predictor |
|---------------|------------------------------|-------------------|----------------------------|
| binary        | Mann–Whitney U               | Spearman ρ        | χ² / Fisher                |
| continuous    | Spearman ρ                   | Spearman ρ        | Kruskal–Wallis             |
| ordinal       | Spearman ρ                   | Spearman ρ        | χ²                         |
| nominal       | Kruskal–Wallis               | χ²                | χ²                         |

`POSITIVE_CLASS` applies only to **binary** targets. Multivariable logistic (§11) remains **binary outcomes only**.

Per-target p-values are corrected with **Benjamini–Hochberg FDR**.


In [ ]:
assoc = screen_associations(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
)

assoc[assoc['fdr_significant']]


In [ ]:
#🟧🟧🟧 Full table
#assoc

## 11. Multiple imputation (MICE)

We generate **m=10** imputed datasets via sklearn's IterativeImputer
(RandomForest estimator, separate random seed per imputation).
The pooled inferential stage applies Rubin's rules over these 10 fits.

For a quick screening run set `m=3`. For publication use `m≥10`.


In [ ]:
M = 3  # number of imputations; reduce to 3 for fast iteration

#imputed_frames = mice_impute(df, schema, m=M, max_iter=10,
#                               random_state=42, output_root=OUTPUT_ROOT)

#print(f"Generated {len(imputed_frames)} imputed frames")

#print("NaN count in first imputed frame:", imputed_frames[0].isna().sum().sum())

## 12. Multivariable logistic regression (Rubin-pooled)

For each target:

1. Build design matrix (continuous z-scored, ordinal kept as codes, nominal one-hot).
2. Iteratively drop predictors with **VIF > 5** to handle collinearity.
3. Fit logistic regression on each of the m imputed frames.
4. Pool coefficients with **Rubin's rules** (Barnard–Rubin df).
5. Report adjusted OR with 95% CI and pooled p-value.
6. Save a forest plot SVG per target.


In [ ]:
#🟧🟧🟧 Skip MICE for now — median/mode imputation (binary left NaN by default)

_df_pre_impute = df.copy()
imputed_frames = [simple_impute(df, schema, impute_binary=False)]

display(imputation_audit(
    _df_pre_impute,
    imputed_frames[0],
    schema,
    INFERENTIAL_PREDICTORS,
    impute_binary=False,
))

print("NaN count (all columns):", imputed_frames[0].isna().sum().sum())
print(
    "cystic_component NaN before:", _df_pre_impute["cystic_component"].isna().sum(),
    "after:", imputed_frames[0]["cystic_component"].isna().sum(),
)


In [ ]:
display(summarize_multivariable_cases(
    imputed_frames[0],
    schema,
    targets=INFERENTIAL_TARGETS,
    predictors=INFERENTIAL_PREDICTORS,
    positive_class=INFERENTIAL_POSITIVE_CLASS,
    vif_threshold=5.0,
))

In [ ]:
inf_results = run_inferential(
    imputed_frames, schema,
    targets=INFERENTIAL_TARGETS,
    predictors=INFERENTIAL_PREDICTORS,
    positive_class=INFERENTIAL_POSITIVE_CLASS,
    vif_threshold=5.0,
    output_root=OUTPUT_ROOT,
)
#inf_results

## 12. Report (§08)

Builds `report.html` from artifacts already in `output/` (DDA, EDA, inferential).
Edit the settings cell, then run both cells.

- **`REPORT_TITLE` / `REPORT_AUTHOR`** — shown on the cover.
- **`REPORT_PATH`** — where to write the HTML file.
- **`FOCUS_PREDICTOR`** — one column gets a dedicated spotlight section (`None` to skip).
- **`FOCUS_REFERENCE_LEVEL`** — reference level for nominal focus vars (or `None`).
- **`analysis_years` / `year_column`** — reuse cohort settings from §02b.


In [ ]:
REPORT_TITLE = "Meningioma analysis report"
REPORT_AUTHOR = ""
REPORT_PATH = OUTPUT_ROOT / "report" / "report.html"

# Spotlight one predictor in the report (None = no focus section)
FOCUS_PREDICTOR = "tumor_episode"
FOCUS_REFERENCE_LEVEL = None


In [ ]:
_c08 = load("08_report_settings")
_c08.run_report(
    df,
    output_root=OUTPUT_ROOT,
    report_title=REPORT_TITLE,
    report_author=REPORT_AUTHOR,
    report_path=REPORT_PATH,
    focus_predictor=FOCUS_PREDICTOR,
    focus_reference_level=FOCUS_REFERENCE_LEVEL,
    analysis_years=ANALYSIS_YEARS,
    year_column=YEAR_COLUMN,
    eda_targets=EDA_TARGETS,
)


## 13. Outputs

Everything is saved under `output/`.


In [ ]:
from pathlib import Path
for p in sorted(Path(OUTPUT_ROOT).rglob('*')):
    if p.is_file():
        print(p)


In [ ]:
#🟧🟧🟧 number of high-grade events

display(df["high_grade"].value_counts(dropna=False))

#🟧🟧🟧 rows actually used in model

print(df.shape)

#🟧🟧🟧 predictors per event rough check

n_events = df["high_grade"].sum()
n_predictors = len(INFERENTIAL_PREDICTORS)
print(n_events / n_predictors)              # ==> <5 means the multivariate model is unstable - decrease the number of variables

## 14. NOTES — Why each statistical choice

Concise but detailed rationale for every formula used in this pipeline.
For each: **what it does**, **why chosen**, **what was rejected**.

---

### Schema inference (hybrid auto + override)

- **What.** Heuristic classification of each column into `continuous / count / ordinal / nominal / binary / datetime / id / text / skip` based on dtype, cardinality, value patterns.
- **Why.** Test selection downstream is kind-driven — a wrong kind silently picks the wrong test (e.g. treating Gleason 1–5 as `continuous` instead of `ordinal` swaps Spearman for MWU and loses interpretability of "per-grade increase").
- **Alternatives rejected.**
  - *Full auto-only*: brittle on clinical data where 0/1-coded ordinals look numeric.
  - *Manual ColSpec per column*: correct but tedious; you'd re-type 30+ specs per study.

---

### Duplicate auditing on normalized string keys

- **What.** Lowercase + strip + empty→NA on ID columns, then flag rows whose full key tuple is non-null and repeated.
- **Why.** Clinical IDs (`pk`, `year`) frequently have invisible whitespace or case drift across data-entry sessions. Naive `duplicated()` misses these.
- **Alternatives rejected.**
  - *Exact match*: under-detects.
  - *Fuzzy match (Levenshtein)*: over-detects, would falsely merge genuinely different patients.

---

### Mann–Whitney U for continuous/count vs binary outcome

- **What.** Non-parametric rank-sum test. H₀: P(X₁ > X₂) = ½. Two-sided.
- **Effect size.** Rank-biserial **r = |Z|/√N**, where Z is the large-sample normal approximation of U. Bounded 0–1, interpretable like Cohen's r (0.1 small, 0.3 medium, 0.5 large).
- **Why.**
  - Clinical continuous variables (PSA, vecums, days-to-surgery) are **almost never normal** — PSA in particular is heavily right-skewed.
  - MWU has ~95% efficiency vs t-test under normality and is far more robust under non-normality.
  - One test for the whole pipeline = no test-switching artifacts.
- **Alternatives rejected.**
  - *Welch's t-test always*: violates assumption on skewed data; inflates type-I error on small skewed samples.
  - *Auto Shapiro-Wilk switch (t if normal, MWU else)*: the normality test itself adds noise and its decision is sample-size dependent (always rejects normal at large N, never at small N) — produces worse calibration than just using MWU.
  - *Welch's t on log-transformed data*: works for PSA specifically but not generalizable to all continuous predictors in the pipeline.
- **Sensitivity.** When publishing, re-run Welch's t on log(PSA) as a sensitivity analysis — if direction and significance agree with MWU, you're robust.

---

### Spearman ρ for ordinal vs binary outcome

- **What.** Pearson correlation on the ranks of category codes vs the 0/1-encoded outcome.
- **Why.**
  - Preserves the **ordering** of ordinal predictors (Gleason 1<2<3<4<5, PIRADS 1<2<3<4<5, risk_group low<mid<high). χ² throws this away — it would only tell you "the distribution differs across levels", not "higher Gleason → more upgrades".
  - Yields a signed, scale-free effect size (ρ) that's directly publishable.
- **Alternatives rejected.**
  - *χ² on the ordinal × binary table*: ignores ordering, weaker power, no direction.
  - *Cochran-Armitage trend test*: equivalent to a linear-trend variant of χ² and gives p only — Spearman gives p **plus** a comparable ρ across all ordinal predictors.
  - *Kendall's τ*: similar info but slower on large N and no power advantage here.

---

### χ² (or Fisher exact) for nominal vs binary

- **What.** χ² of independence on the contingency table, **without Yates correction** (modern recommendation — Yates is overconservative). Switches to **Fisher exact** if the 2×2 table has any expected cell count < 5.
- **Why Fisher when expected<5.** χ²'s asymptotic distribution breaks down with small expected counts; Fisher's exact test conditions on the marginals and computes the exact hypergeometric p — correct at any sample size.
- **Effect size: Cramér's V** = √(χ²/(N·(min(r,c)−1))). Bounded 0–1, comparable across table shapes. For 2×2 tables we **also** report the odds ratio because clinicians read OR natively.
- **Alternatives rejected.**
  - *Yates-corrected χ²*: too conservative for modern computing — Fisher is exact and almost as fast.
  - *G-test (likelihood ratio)*: theoretically nicer for nested models but identical conclusions in 2-way tables; less familiar to clinical readers.
  - *Permutation χ²*: same answer as Fisher for 2×2, more expensive.

---

### Benjamini–Hochberg FDR correction, per target

- **What.** Sort p-values ascending; for rank i out of m, compute q_i = p_(i)·m/i; enforce monotonicity from the right; significance at q < α controls expected proportion of false discoveries at α.
- **Why per-target (not pooled across all targets).** Each outcome (upgrade, upstage, downgrade) is a **separate family** of hypotheses with its own scientific interpretation. Pooling them inflates the family size and over-corrects. This matches how clinical journals report multi-outcome studies.
- **Alternatives rejected.**
  - *Bonferroni*: controls family-wise error rate — far too conservative for a screening stage with 10+ predictors. Misses real signal.
  - *Holm-Bonferroni*: still FWER, marginally less conservative than Bonferroni but still much stricter than BH.
  - *Storey q-value*: estimates the null proportion adaptively; great when you have hundreds of tests but unstable at small m (you'll have <20 tests per target).
  - *No correction*: indefensible with ≥3 predictors per target — false discovery rate would be ~30%+.
- **Verified.** Output matches `statsmodels.stats.multitest.multipletests(method='fdr_bh')` exactly.

---

### MICE (Multiple Imputation by Chained Equations), m=10

- **What.** For each missing value: fit a regression of that column on all others using observed data, predict missing values, iterate until convergence. Repeat with m different random seeds to produce m plausible completed datasets.
- **Why multiple (not single).** Single imputation pretends the imputed values are known, so it **understates standard errors**. With m=10 imputations and Rubin pooling, the SEs honestly include imputation uncertainty.
- **Estimator: RandomForestRegressor.** Captures non-linear relationships (PSA × vecums × Gleason interactions) without you specifying them. Tolerates mixed numeric/categorical inputs.
- **Why m=10.** Rubin showed efficiency = (1 + fmi/m)^(-1) where fmi is fraction of missing info. At fmi ≈ 0.3 (typical clinical data), m=10 gives ~97% efficiency. m=5 is acceptable, m=20 is overkill.
- **Alternatives rejected.**
  - *Mean/median imputation*: distorts variance and any correlation involving the imputed column. Catastrophic for inferential SEs.
  - *Complete-case analysis*: throws away rows with any missingness — typically 20–50% data loss in clinical cohorts; introduces selection bias if missingness is MAR (which it usually is).
  - *Hot-deck imputation*: works for nominal-only data; weaker for mixed types.
  - *Bayesian model-based imputation (`mice` R package, Stan)*: gold standard but heavy infrastructure; sklearn's `IterativeImputer` is close enough for clinical screening.
- **Limitation.** Assumes data is **Missing At Random** (MAR) — missingness depends only on observed variables. For **MNAR** patterns (e.g. "PSA was missing because risk was low"), add explicit `<col>_missing` flags in section 6a so the model can use the missingness indicator itself as a predictor.

---

### Missingness flags

- **What.** Binary indicator columns `<col>_missing` added before imputation.
- **Why.** In clinical data, *that a value was missing* is often informative (e.g. PSA not measured because clinician judged it unnecessary). Including the flag in the regression lets the model separate "the value's effect" from "the act of measuring's effect".
- **Alternatives rejected.**
  - *Imputing without flags*: hides the MNAR mechanism.
  - *Dropping columns with high missingness*: throws away signal; missingness % is not a reliable filter for clinical utility.

---

### Variance Inflation Factor (VIF) pruning, threshold = 5

- **What.** For each predictor x_j, VIF = 1/(1 − R²_j), where R²_j is from regressing x_j on all other predictors. Iteratively drop the column with the highest VIF until all ≤ 5.
- **Why.** Logistic regression with collinear predictors produces enormous standard errors and unstable coefficients ("model can't tell whether PSA or PSA-density is doing the work"). VIF > 5 ⇔ R²_j > 0.80 ⇔ severe multicollinearity.
- **Why threshold = 5** (not 10). VIF=10 is the classical statistics teaching threshold but for clinical regression with modest N (<500), 5 is the modern recommendation (Vatcheva 2016, O'Brien 2007).
- **Alternatives rejected.**
  - *Pairwise Pearson correlation > 0.8*: catches only 2-variable collinearity; misses 3-way (e.g. a = b + c).
  - *Lasso regularization*: would drop collinear features automatically but **biases coefficients** toward zero — bad for inference (you want unbiased OR estimates). Lasso is for prediction, not inference.
  - *Ridge / Elastic Net*: same problem — shrinks coefficients, distorts ORs.
  - *PCA / partial-least-squares*: components are uninterpretable clinically.

---

### Multivariable logistic regression

- **What.** Per target, one binary logistic model with all surviving predictors. Continuous z-scored (so OR is per-SD increase), ordinals kept as numeric codes, nominals one-hot with drop_first.
- **Why.** Univariate screening (section 10) ignores confounding — Gleason can show up "significant" purely because it correlates with PSA. Multivariable estimates the **adjusted** effect of each predictor holding the others constant.
- **Why this design encoding.**
  - *z-score continuous*: ORs comparable across predictors; one "unit" = one SD.
  - *Ordinal as numeric code*: assumes linear log-odds across levels (parsimonious; standard for Gleason/PIRADS in urology papers). The alternative is one-hot with drop_first, which uses more degrees of freedom and is only worth it if the trend is clearly non-monotonic — check the EDA plots first.
  - *Nominal one-hot drop_first*: avoids the dummy variable trap (perfect collinearity with intercept).
- **Alternatives rejected.**
  - *Univariate-only pipeline*: misleading because of confounding.
  - *Stepwise selection (forward/backward)*: notorious for unstable selection, inflated significance, and irreproducibility. Modern guidance (Harrell, Steyerberg) is: don't.
  - *Random forest / XGBoost*: better predictive accuracy but no clinical OR with CI to report.
  - *Penalized regression (Firth, Lasso, Ridge)*: useful with extreme separation or n<<p but biases the OR estimates — defeats the inferential purpose.
  - *Bayesian logistic with weakly informative priors*: cleaner for tiny samples and would give credible intervals — but you'd need to defend prior choice in the manuscript.

---

### Rubin's rules with Barnard–Rubin degrees of freedom

- **What.** Across the m=10 imputed-frame fits, for each coefficient:
  - θ̄ = mean of the m point estimates
  - within-imp variance Ū = mean of the m squared SEs
  - between-imp variance B = sample variance of the m estimates
  - total variance T = Ū + (1 + 1/m)·B
  - pooled SE = √T
  - degrees of freedom (Barnard–Rubin):
    df = (m−1)·(1 + Ū/((1+1/m)·B))²
  - p-value from t-distribution with that df; 95% CI = θ̄ ± t_{0.975, df}·SE
- **Why.** Rubin's rules are the **only** statistically valid way to combine results across multiple imputations. The total variance T splits into "within" (each model's uncertainty) and "between" (uncertainty due to missing data) — they're not interchangeable.
- **Why Barnard–Rubin df (not the original Rubin 1987 df).** Original Rubin df → ∞ when between-variance is small, which is wrong when m is small. Barnard–Rubin (1999) is a small-sample correction that's now the standard (R `mice` uses it, SAS PROC MIANALYZE uses it).
- **Alternatives rejected.**
  - *Picking the "best" imputation*: defeats the purpose of multiple imputation entirely.
  - *Average the imputed datasets first, then fit once*: produces correct point estimates but **wrong SEs** (the between-variance is invisible).
  - *Use the within-variance only*: ignores imputation uncertainty — false confidence.
- **Verified.** With zero between-variance, our pooler returns SE equal to the single-fit SE; with non-zero between, it correctly inflates SE and produces a finite small-sample df (e.g. m=5, modest B → df ≈ 22).

---

### Why log-scale x-axis on forest plots

- ORs are multiplicative (OR=2 and OR=0.5 are equal-and-opposite effects). On a linear axis they look asymmetric; on log scale they're symmetric around OR=1, which is the correct visual.

---

### What this pipeline deliberately does NOT do

- **No machine-learning prediction** (no train/test split, no AUC, no calibration). This is an **association/inference** pipeline, not a prediction pipeline. If you later want a predictive model (e.g. nomogram for upgrade risk), that's a separate workflow with cross-validation, calibration plots, decision-curve analysis.
- **No causal inference** (no DAGs, no IPTW, no instrumental variables). All effects here are **statistical associations** adjusted for the included covariates — they are *not* causal effects. Manuscript wording must say "associated with", never "causes".
- **No survival/time-to-event analysis.** Targets here are binary (upgrade yes/no). If you later care about *time to biochemical recurrence*, you'd need Cox regression — a separate module.

---

### Sanity-check checklist before submitting results

1. Print `schema_summary(schema)` — every ordinal has correct `ordered_levels`?
2. After MICE, `imputed_frames[0].isna().sum().sum()` == 0 for predictor columns?
3. `inf_results['n_models']` ≈ m for all predictors (means the model converged on every imputation)?
4. Forest plot ORs and EDA univariate effects agree in **direction** (sign)? If they flip, you have confounding worth discussing.
5. For each FDR-significant univariate result, check the corresponding plot in `output/eda/figures/` — is the pattern visually credible or driven by 2–3 outliers?
